# EarthBridge Kaggle Baseline Training

Use this notebook for free online training on the Kaggle BEN-14K dataset. It expects `/kaggle/input/datasets/narendraaironi/bigearthnet-14k/BEN_14k`, then exports `artifacts/earthbridge_export.zip` for the local demo.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = ""  # Optional: set to your GitHub repo URL.
ROOT = Path("/kaggle/working/EarthBridge")

if REPO_URL and not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

if not ROOT.exists():
    ROOT = Path.cwd()

os.chdir(ROOT)
print("Working directory:", ROOT)

In [ ]:
!python -m pip install -q -r requirements.txt
!python -m pip install -q faiss-cpu rasterio albumentations opencv-python matplotlib tensorboard fastapi "uvicorn[standard]" python-multipart
!python -m pip install -q -e .

## Configure Dataset

The default path targets Kaggle BEN-14K. `BigEarthNet-S1` is treated as SAR and `BigEarthNet-S2` is treated as multispectral.

In [ ]:
DATA_RAW = Path("/kaggle/input/datasets/narendraaironi/bigearthnet-14k/BEN_14k")
IMAGE_ROOT = DATA_RAW.parent
LEFT_MODALITY = "multispectral"
RIGHT_MODALITY = "sar"

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("DATA_RAW:", DATA_RAW)
print("IMAGE_ROOT:", IMAGE_ROOT)
print("PAIR:", LEFT_MODALITY, "->", RIGHT_MODALITY)
print("DEVICE:", DEVICE)
assert DATA_RAW.exists(), f"Dataset path does not exist: {DATA_RAW}"

In [ ]:
!python scripts/run_cloud_pipeline.py \
  --data-raw "{DATA_RAW}" \
  --image-root "{IMAGE_ROOT}" \
  --left-modality "{LEFT_MODALITY}" \
  --right-modality "{RIGHT_MODALITY}" \
  --image-size 128 \
  --batch-size 16 \
  --epochs 5 \
  --device "{DEVICE}" \
  --top-k 10 \
  --export-zip artifacts/earthbridge_export.zip
print("Download this file from Kaggle:", ROOT / "artifacts/earthbridge_export.zip")